In [0]:
# ============================================================
# CÉLULA 1 — SETUP
# DQ MONITORING LOGS - ITENS_PEDIDO e RASTREAMENTO_ENTREGAS
# ============================================================

import uuid
from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql.window import Window

adls_client_id = dbutils.secrets.get(scope="internship-squad1", key="adls-client-id")
adls_tenant_id = dbutils.secrets.get(scope="internship-squad1", key="adls-tenant-id")
adls_client_secret = dbutils.secrets.get(scope="internship-squad1", key="adls-client-secret")
adls_storage_account = dbutils.secrets.get(scope="internship-squad1", key="adls-storage-account")

adls_options = {
    f"fs.azure.account.auth.type.{adls_storage_account}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{adls_storage_account}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{adls_storage_account}.dfs.core.windows.net": adls_client_id,
    f"fs.azure.account.oauth2.client.secret.{adls_storage_account}.dfs.core.windows.net": adls_client_secret,
    f"fs.azure.account.oauth2.client.endpoint.{adls_storage_account}.dfs.core.windows.net": f"https://login.microsoftonline.com/{adls_tenant_id}/oauth2/token",
}

base_squad1 = f"abfss://squad1@{adls_storage_account}.dfs.core.windows.net"

caminhos = {
    "silver_itens_pedido": f"{base_squad1}/dupla1/silver/ecommerce_itens_pedido",
    "silver_rastreamento": f"{base_squad1}/dupla1/silver/ecommerce_rastreamento_entregas",
    "silver_pedidos": f"{base_squad1}/silver/ecommerce_pedidos",
    "dq_logs": f"{base_squad1}/dq_monitoring_logs",
}

print("Credenciais OAuth preparadas.")
print("DQ Logs (compartilhado):", caminhos["dq_logs"])

In [0]:
# ============================================================
# CÉLULA 2 — REGRAS DE QUALIDADE POR LINHA (com severidade)
# ============================================================

# itens_pedido: colunas booleanas já gravadas na Silver
REGRAS_ITENS_PEDIDO = {
    "tecnica_1_pk_valida":              "Crítica",
    "tecnica_2_fk_pedido_valido":       "Crítica",
    "tecnica_3_fk_sku_valido":          "Crítica",
    "tecnica_4_quantidade_valida":      "Crítica",
    "tecnica_5_preco_valido":           "Crítica",
    "negocio_6_desconto_nao_maior_preco": "Aviso",
    "negocio_8_desconto_nao_negativo":    "Aviso",
    "negocio_9_desconto_dentro_limite":   "Aviso",
}

# rastreamento_entregas: colunas booleanas já gravadas na Silver
REGRAS_RASTREAMENTO = {
    "tecnica_1_pk_valida":              "Crítica",
    "tecnica_2_status_valido":          "Crítica",
    "tecnica_3_fk_pedido_valido":       "Crítica",
    "tecnica_4_data_valida":            "Crítica",
    "tecnica_5_codigo_rastreio_valido": "Crítica",
    "negocio_6_ordem_cronologica":      "Aviso",
}

print("Regras itens_pedido:", list(REGRAS_ITENS_PEDIDO.keys()))
print("Regras rastreamento:", list(REGRAS_RASTREAMENTO.keys()))

In [0]:
# ============================================================
# CÉLULA 3 — CALCULAR FALHAS POR REGRA (linhas)
# ============================================================

run_id = str(uuid.uuid4())
agora = datetime.now(timezone.utc)

def calcular_falhas_por_regra(df, nome_tabela, regras_dict):
    total = df.count()
    linhas = []
    for regra, severidade in regras_dict.items():
        qtd_falhas = df.filter(F.col(regra) == False).count()
        status = "FAIL" if qtd_falhas > 0 else "PASS"
        linhas.append(
            (run_id, nome_tabela, regra, status, severidade, qtd_falhas, total, agora)
        )
        print(f"[{nome_tabela}] {regra} -> {status} | falhas: {qtd_falhas}/{total}")
    return linhas

df_silver_itens = spark.read.format("delta").options(**adls_options).load(caminhos["silver_itens_pedido"])
df_silver_rastreamento = spark.read.format("delta").options(**adls_options).load(caminhos["silver_rastreamento"])

linhas_logs = []
linhas_logs += calcular_falhas_por_regra(df_silver_itens, "ecommerce_itens_pedido", REGRAS_ITENS_PEDIDO)
linhas_logs += calcular_falhas_por_regra(df_silver_rastreamento, "ecommerce_rastreamento_entregas", REGRAS_RASTREAMENTO)

In [0]:
# ============================================================
# CÉLULA 4 — REGRAS AGREGADAS (por pedido)
# ============================================================

df_silver_pedidos = spark.read.format("delta").options(**adls_options).load(caminhos["silver_pedidos"])

# --- ITENS_PEDIDO: Regra 7 (soma dos itens == valor_total do pedido) ---
soma_por_pedido = (df_silver_itens
    .withColumn("valor_item", (F.col("preco_unitario") - F.col("desconto_aplicado")) * F.col("quantidade"))
    .groupBy("id_pedido").agg(F.sum("valor_item").alias("soma_calculada"))
    .join(df_silver_pedidos.select("id_pedido", "valor_total"), on="id_pedido", how="left"))

total_pedidos_itens = soma_por_pedido.count()
divergentes = soma_por_pedido.filter(
    F.col("valor_total").isNull() | (F.abs(F.col("soma_calculada") - F.col("valor_total")) > 0.01)
).count()
status = "FAIL" if divergentes > 0 else "PASS"
linhas_logs.append((run_id, "ecommerce_itens_pedido", "negocio_7_soma_itens_igual_valor_total", status, "Aviso", divergentes, total_pedidos_itens, agora))
print(f"[itens_pedido] negocio_7_soma_itens_igual_valor_total -> {status} | falhas: {divergentes}/{total_pedidos_itens}")

# --- ITENS_PEDIDO: Regra 10 (todo pedido tem >=1 item) ---
total_pedidos = df_silver_pedidos.count()
pedidos_com_item = df_silver_itens.select("id_pedido").distinct()
pedidos_sem_item = df_silver_pedidos.select("id_pedido").distinct().subtract(pedidos_com_item).count()
status = "FAIL" if pedidos_sem_item > 0 else "PASS"
linhas_logs.append((run_id, "ecommerce_itens_pedido", "negocio_10_pedido_tem_item", status, "Aviso", pedidos_sem_item, total_pedidos, agora))
print(f"[itens_pedido] negocio_10_pedido_tem_item -> {status} | falhas: {pedidos_sem_item}/{total_pedidos}")

# --- RASTREAMENTO: Regra 7 (pedido Entregue tem evento 'entregue') ---
ids_com_evento_entregue = set(r["id_pedido_ecommerce"] for r in
    df_silver_rastreamento.filter(F.col("status_entrega") == "entregue").select("id_pedido_ecommerce").distinct().collect())
pedidos_entregues = df_silver_pedidos.filter(F.col("status_pedido") == "Entregue").select("id_pedido").distinct()
total_entregues = pedidos_entregues.count()
sem_evento_entregue = pedidos_entregues.filter(~F.col("id_pedido").isin(ids_com_evento_entregue)).count()
status = "FAIL" if sem_evento_entregue > 0 else "PASS"
linhas_logs.append((run_id, "ecommerce_rastreamento_entregas", "negocio_7_entregue_tem_evento", status, "Aviso", sem_evento_entregue, total_entregues, agora))
print(f"[rastreamento] negocio_7_entregue_tem_evento -> {status} | falhas: {sem_evento_entregue}/{total_entregues}")

# --- RASTREAMENTO: Regra 8 (<=30 dias entre coletado e entregue) ---
eventos_coletado = df_silver_rastreamento.filter(F.col("status_entrega") == "coletado").groupBy("id_pedido_ecommerce").agg(F.min("dt_evento").alias("dt_coletado"))
eventos_entregue = df_silver_rastreamento.filter(F.col("status_entrega") == "entregue").select("id_pedido_ecommerce", F.col("dt_evento").alias("dt_entregue"))
prazos = eventos_coletado.join(eventos_entregue, on="id_pedido_ecommerce", how="inner").withColumn("dias_entrega", F.datediff("dt_entregue", "dt_coletado"))
total_prazos = prazos.count()
fora_do_prazo = prazos.filter(F.col("dias_entrega") > 30).count()
status = "FAIL" if fora_do_prazo > 0 else "PASS"
linhas_logs.append((run_id, "ecommerce_rastreamento_entregas", "negocio_8_prazo_entrega_30_dias", status, "Aviso", fora_do_prazo, total_prazos, agora))
print(f"[rastreamento] negocio_8_prazo_entrega_30_dias -> {status} | falhas: {fora_do_prazo}/{total_prazos}")

# --- RASTREAMENTO: Regra 9 (pedido cancelado não pode ter evento) ---
pedidos_cancelados = df_silver_pedidos.filter(F.col("status_pedido") == "Cancelado").select("id_pedido").distinct()
total_cancelados = pedidos_cancelados.count()
ids_cancelados_com_evento = (df_silver_rastreamento.select("id_pedido_ecommerce").distinct()
    .join(pedidos_cancelados.withColumnRenamed("id_pedido", "id_pedido_ecommerce"), on="id_pedido_ecommerce", how="inner").count())
status = "FAIL" if ids_cancelados_com_evento > 0 else "PASS"
linhas_logs.append((run_id, "ecommerce_rastreamento_entregas", "negocio_9_cancelado_sem_evento", status, "Aviso", ids_cancelados_com_evento, total_cancelados, agora))
print(f"[rastreamento] negocio_9_cancelado_sem_evento -> {status} | falhas: {ids_cancelados_com_evento}/{total_cancelados}")

# --- RASTREAMENTO: Regra 10 (pedido não cancelado tem só 1 transportadora) ---
ids_cancelados_geral = set(r["id_pedido"] for r in pedidos_cancelados.collect())
transportadoras_por_pedido = (df_silver_rastreamento
    .filter(~F.col("id_pedido_ecommerce").isin(ids_cancelados_geral))
    .groupBy("id_pedido_ecommerce").agg(F.countDistinct("id_transportadora").alias("qtd_transportadoras")))
total_nao_cancelados = transportadoras_por_pedido.count()
inconsistentes = transportadoras_por_pedido.filter(F.col("qtd_transportadoras") > 1).count()
status = "FAIL" if inconsistentes > 0 else "PASS"
linhas_logs.append((run_id, "ecommerce_rastreamento_entregas", "negocio_10_transportadora_unica", status, "Aviso", inconsistentes, total_nao_cancelados, agora))
print(f"[rastreamento] negocio_10_transportadora_unica -> {status} | falhas: {inconsistentes}/{total_nao_cancelados}")

print(f"\nTotal de regras no log: {len(linhas_logs)}")

In [0]:
# ============================================================
# CÉLULA 5 — MONTAR SCHEMA E DATAFRAME DE LOGS
# ============================================================

from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, TimestampType
)

schema_logs = StructType([
    StructField("run_id", StringType(), False),
    StructField("tabela", StringType(), False),
    StructField("regra", StringType(), False),
    StructField("status", StringType(), False),
    StructField("severidade", StringType(), False),
    StructField("qtd_registros_falhos", LongType(), False),
    StructField("qtd_registros_total", LongType(), False),
    StructField("timestamp_execucao", TimestampType(), False),
])

df_novo = spark.createDataFrame(linhas_logs, schema=schema_logs)

print("===== LOGS QUE SERÃO GRAVADOS =====")
display(df_novo)

In [0]:
# ============================================================
# CÉLULA 6 — GRAVAR NO DQ_MONITORING_LOGS (COMPARTILHADO)
# ============================================================

(df_novo.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .options(**adls_options)
    .save(caminhos["dq_logs"]))

print(f"[OK] {len(linhas_logs)} logs gravados em dq_monitoring_logs (run_id={run_id}).")

In [0]:
# ============================================================
# CÉLULA 7 — VALIDAÇÃO FINAL
# ============================================================

df_logs_final = spark.read.format("delta").options(**adls_options).load(caminhos["dq_logs"])

df_nossos_logs = (df_logs_final
    .filter(F.col("run_id") == run_id)
    .orderBy("tabela", "regra"))

print("===== NOSSOS LOGS NESTA EXECUÇÃO =====")
display(df_nossos_logs)

print(f"\nTotal de linhas na tabela compartilhada dq_monitoring_logs: {df_logs_final.count()}")